# SEED Simple MoE Parameter Search

This notebook trains only `SimpleMoEClassifier`, searches a small hyperparameter grid, and writes validation-weighted voting predictions to `SEED.txt`.

Runtime policy follows `train_seed_simple_models.ipynb`: CUDA is used when available, and the final submission is produced by voting across the different MoE parameter settings.

In [9]:
import json
import os
import random
from pathlib import Path

import h5py
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from course_project.TEST_DATASET import TestDataset, TrainDataset

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())

torch: 2.7.0
cuda available: True


## 1) Configuration

Run all cells to perform the full search. For a quick smoke test, set `MAX_CONFIGS = 1` and `EPOCHS = 2`.

In [10]:
DATA_NAME = "SEED"
DATA_DIR = Path("G:/MLproject/course_project") / DATA_NAME
TRAIN_PATH = DATA_DIR / "train.h5"
VAL_PATH = DATA_DIR / "val.h5"
TEST_PATH = DATA_DIR / "test_x_only.h5"

OUTPUT_PATH = DATA_DIR / "SEED.txt"
BEST_CHECKPOINT_PATH = DATA_DIR / "best_seed_simple_moe_search.pt"
RESULTS_JSON_PATH = DATA_DIR / "simple_moe_search_results.json"

SEED = 3407
CHANNELS = 62
CLASSES = 3
BATCH_SIZE = 32
EPOCHS = 160
PATIENCE = 24
MIN_DELTA = 1e-4
MIN_LR = 1e-5
GRAD_CLIP = 1.0
ENSEMBLE_MIN_VAL_ACC = 0.42
ENSEMBLE_WEIGHT_POWER = 2.5
MAX_CONFIGS = None

USE_CUDA = torch.cuda.is_available()
device = torch.device("cuda" if USE_CUDA else "cpu")
print("device:", device)

device: cuda


## 1b) GPU/cuDNN Diagnostic

This confirms CUDA convolution works before the long search starts.


In [11]:
if USE_CUDA:
    diag_model = nn.Sequential(
        nn.Conv1d(CHANNELS, 8, kernel_size=7, padding=3, bias=False),
        nn.BatchNorm1d(8),
        nn.GELU(),
        nn.AdaptiveAvgPool1d(1),
        nn.Flatten(),
        nn.Linear(8, CLASSES),
    ).to(device)
    diag_x = torch.randn(4, CHANNELS, 400, device=device)
    diag_y = torch.randint(0, CLASSES, (4,), device=device)
    diag_loss = nn.CrossEntropyLoss()(diag_model(diag_x), diag_y)
    diag_loss.backward()
    torch.cuda.synchronize()
    print("GPU Conv1d backward diagnostic OK:", float(diag_loss.detach().cpu()))
else:
    print("CUDA is not active. Check your notebook kernel.")


GPU Conv1d backward diagnostic OK: 1.3502609729766846


## 2) Helpers and Model

In [12]:
def set_global_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def read_h5_y(path):
    with h5py.File(path, "r") as f:
        return f["y"][()].astype(np.int64)


def accuracy(y_true, y_pred):
    return float(np.mean(np.asarray(y_true) == np.asarray(y_pred)))


def write_labels(path, labels):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for label in labels:
            f.write(f"{int(label)}\n")


def eeg_stat_features(x, eps=1e-6):
    mean = x.mean(dim=-1)
    std = x.std(dim=-1, unbiased=False)
    rms = torch.sqrt(torch.mean(x.square(), dim=-1) + eps)
    peak_to_peak = x.amax(dim=-1) - x.amin(dim=-1)
    return torch.cat([mean, std, rms, peak_to_peak], dim=1)


class SimpleMoEClassifier(nn.Module):
    def __init__(self, chans=CHANNELS, num_classes=CLASSES, hidden_dim=48, num_experts=4, dropout=0.25):
        super().__init__()
        kernels = (3, 7, 15, 31)[:num_experts]
        self.experts = nn.ModuleList(
            [
                nn.Sequential(
                    nn.Conv1d(chans, hidden_dim, kernel_size=kernel, padding=kernel // 2, bias=False),
                    nn.BatchNorm1d(hidden_dim),
                    nn.GELU(),
                    nn.Dropout(dropout),
                    nn.Conv1d(hidden_dim, hidden_dim, kernel_size=3, padding=1, groups=hidden_dim, bias=False),
                    nn.BatchNorm1d(hidden_dim),
                    nn.GELU(),
                    nn.AdaptiveAvgPool1d(1),
                    nn.Flatten(),
                    nn.Linear(hidden_dim, num_classes),
                )
                for kernel in kernels
            ]
        )
        self.router = nn.Sequential(
            nn.Linear(chans * 4, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, len(self.experts)),
        )

    def forward(self, x):
        route_logits = self.router(eeg_stat_features(x))
        route_weights = torch.softmax(route_logits, dim=1)
        expert_logits = torch.stack([expert(x) for expert in self.experts], dim=1)
        return torch.sum(route_weights.unsqueeze(-1) * expert_logits, dim=1)

    def clip_gradients(self, max_norm=1.0):
        torch.nn.utils.clip_grad_norm_(self.parameters(), max_norm)

## 3) Search Grid

Fine search around the current best `moe_h64_dp30` configuration.

In [13]:
def build_search_configs(data_dir):
    data_dir = Path(data_dir)
    specs = [
        ("moe_h64_dp30_center", 64, 0.30, 6e-4, 1e-3, 0.05, 6),
        ("moe_h56_dp30", 56, 0.30, 6e-4, 1e-3, 0.05, 6),
        ("moe_h72_dp30", 72, 0.30, 6e-4, 1e-3, 0.05, 6),
        ("moe_h64_dp27", 64, 0.27, 6e-4, 1e-3, 0.05, 6),
        ("moe_h64_dp33", 64, 0.33, 6e-4, 1e-3, 0.05, 6),
        ("moe_h64_dp30_lr55", 64, 0.30, 5.5e-4, 1e-3, 0.05, 6),
        ("moe_h64_dp30_lr65", 64, 0.30, 6.5e-4, 1e-3, 0.05, 6),
        ("moe_h64_dp30_wd08_ls04", 64, 0.30, 6e-4, 8e-4, 0.04, 6),
        ("moe_h64_dp30_wd12_ls06", 64, 0.30, 6e-4, 1.2e-3, 0.06, 6),
        ("moe_h60_dp28_lr58", 60, 0.28, 5.8e-4, 9e-4, 0.05, 6),
        ("moe_h68_dp32_lr62", 68, 0.32, 6.2e-4, 1.1e-3, 0.05, 6),
    ]
    return [
        {
            "name": name,
            "hidden_dim": hidden_dim,
            "num_experts": 4,
            "dropout": dropout,
            "lr": lr,
            "weight_decay": weight_decay,
            "label_smoothing": label_smoothing,
            "scheduler_patience": scheduler_patience,
            "checkpoint": data_dir / f"best_seed_simple_moe_search_{name}.pt",
        }
        for name, hidden_dim, dropout, lr, weight_decay, label_smoothing, scheduler_patience in specs
    ]


def select_best_result(results):
    if not results:
        raise ValueError("No search results were produced.")
    return max(enumerate(results), key=lambda item: (item[1]["val_acc"], -item[0]))[1]


search_configs = build_search_configs(DATA_DIR)
if MAX_CONFIGS is not None:
    search_configs = search_configs[:MAX_CONFIGS]

for config in search_configs:
    print(config["name"], {k: v for k, v in config.items() if k != "checkpoint"})

moe_h64_dp30_center {'name': 'moe_h64_dp30_center', 'hidden_dim': 64, 'num_experts': 4, 'dropout': 0.3, 'lr': 0.0006, 'weight_decay': 0.001, 'label_smoothing': 0.05, 'scheduler_patience': 6}
moe_h56_dp30 {'name': 'moe_h56_dp30', 'hidden_dim': 56, 'num_experts': 4, 'dropout': 0.3, 'lr': 0.0006, 'weight_decay': 0.001, 'label_smoothing': 0.05, 'scheduler_patience': 6}
moe_h72_dp30 {'name': 'moe_h72_dp30', 'hidden_dim': 72, 'num_experts': 4, 'dropout': 0.3, 'lr': 0.0006, 'weight_decay': 0.001, 'label_smoothing': 0.05, 'scheduler_patience': 6}
moe_h64_dp27 {'name': 'moe_h64_dp27', 'hidden_dim': 64, 'num_experts': 4, 'dropout': 0.27, 'lr': 0.0006, 'weight_decay': 0.001, 'label_smoothing': 0.05, 'scheduler_patience': 6}
moe_h64_dp33 {'name': 'moe_h64_dp33', 'hidden_dim': 64, 'num_experts': 4, 'dropout': 0.33, 'lr': 0.0006, 'weight_decay': 0.001, 'label_smoothing': 0.05, 'scheduler_patience': 6}
moe_h64_dp30_lr55 {'name': 'moe_h64_dp30_lr55', 'hidden_dim': 64, 'num_experts': 4, 'dropout': 0.3,

## 4) DataLoaders

The test loader uses `shuffle=False`, preserving submission order.

In [14]:
set_global_seed(SEED)

train_y = read_h5_y(TRAIN_PATH)
val_y = read_h5_y(VAL_PATH)

train_ds = TrainDataset(str(TRAIN_PATH))
val_ds = TrainDataset(str(VAL_PATH))
test_ds = TestDataset(str(TEST_PATH))

train_generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=train_generator,
    pin_memory=USE_CUDA,
)
train_eval_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=False, pin_memory=USE_CUDA)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, pin_memory=USE_CUDA)
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False, pin_memory=USE_CUDA)

print("train samples:", len(train_ds), "val samples:", len(val_ds), "test samples:", len(test_ds))
print("train label counts:", dict(zip(*np.unique(train_y, return_counts=True))))
print("val label counts:", dict(zip(*np.unique(val_y, return_counts=True))))

train samples: 1000 val samples: 350 test samples: 450
train label counts: {np.int64(0): np.int64(334), np.int64(1): np.int64(333), np.int64(2): np.int64(333)}
val label counts: {np.int64(0): np.int64(116), np.int64(1): np.int64(117), np.int64(2): np.int64(117)}


## 5) Training and Evaluation Functions

In [15]:
def collect_prob(model, loader):
    model.eval()
    probs = []
    with torch.no_grad():
        for batch in loader:
            data = batch[0] if isinstance(batch, (tuple, list)) else batch
            data = data.to(device, dtype=torch.float32, non_blocking=USE_CUDA)
            probs.append(torch.softmax(model(data), dim=1).cpu().numpy())
    return np.concatenate(probs, axis=0)


def train_one_config(config, config_index):
    set_global_seed(SEED)
    checkpoint_path = Path(config["checkpoint"])
    print("\n" + "=" * 72)
    print(f"Training Simple MoE config: {config['name']}")
    print("config:", {k: v for k, v in config.items() if k != "checkpoint"})

    model = SimpleMoEClassifier(
        chans=CHANNELS,
        num_classes=CLASSES,
        hidden_dim=config["hidden_dim"],
        num_experts=config["num_experts"],
        dropout=config["dropout"],
    ).to(device)
    criterion = nn.CrossEntropyLoss(label_smoothing=config["label_smoothing"])
    optimizer = torch.optim.AdamW(model.parameters(), lr=config["lr"], weight_decay=config["weight_decay"])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=config["scheduler_patience"],
        min_lr=MIN_LR,
    )

    best_val_acc = -1.0
    best_epoch = 0
    bad_epochs = 0
    checkpoint_path.parent.mkdir(parents=True, exist_ok=True)

    for epoch in range(1, EPOCHS + 1):
        model.train()
        train_loss_sum = train_correct = train_num = 0
        for data, label in train_loader:
            data = data.to(device, dtype=torch.float32, non_blocking=USE_CUDA)
            label = label.to(device, dtype=torch.long, non_blocking=USE_CUDA)
            optimizer.zero_grad(set_to_none=True)
            output = model(data)
            loss = criterion(output, label)
            loss.backward()
            model.clip_gradients(GRAD_CLIP)
            optimizer.step()

            batch_size = label.size(0)
            train_loss_sum += loss.item() * batch_size
            train_correct += (torch.argmax(output.detach(), dim=1) == label).sum().item()
            train_num += batch_size

        model.eval()
        val_loss_sum = val_correct = val_num = 0
        with torch.no_grad():
            for val_data, val_label in val_loader:
                val_data = val_data.to(device, dtype=torch.float32, non_blocking=USE_CUDA)
                val_label = val_label.to(device, dtype=torch.long, non_blocking=USE_CUDA)
                val_output = model(val_data)
                val_loss = criterion(val_output, val_label)
                batch_size = val_label.size(0)
                val_loss_sum += val_loss.item() * batch_size
                val_correct += (torch.argmax(val_output, dim=1) == val_label).sum().item()
                val_num += batch_size

        epoch_train_loss = train_loss_sum / train_num
        epoch_train_acc = train_correct / train_num
        epoch_val_loss = val_loss_sum / val_num
        epoch_val_acc = val_correct / val_num
        scheduler.step(epoch_val_acc)

        if epoch_val_acc > best_val_acc + MIN_DELTA:
            best_val_acc = epoch_val_acc
            best_epoch = epoch
            bad_epochs = 0
            torch.save(model.state_dict(), checkpoint_path)
        else:
            bad_epochs += 1

        print(
            f"Epoch [{epoch:03d}/{EPOCHS}] | "
            f"Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc:.4f} | "
            f"Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc:.4f} | "
            f"Best: {best_val_acc:.4f} @ {best_epoch:03d}"
        )
        if bad_epochs >= PATIENCE:
            print(f"Early stopping at epoch {epoch}; best epoch was {best_epoch}.")
            break

    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    train_prob = collect_prob(model, train_eval_loader)
    val_prob = collect_prob(model, val_loader)
    test_prob = collect_prob(model, test_loader)

    train_acc = accuracy(train_y, np.argmax(train_prob, axis=1))
    val_acc = accuracy(val_y, np.argmax(val_prob, axis=1))
    checkpoint_labels = np.argmax(test_prob, axis=1).astype(int).tolist()
    assert len(checkpoint_labels) == len(test_ds), f"Prediction count {len(checkpoint_labels)} != test sample count {len(test_ds)}"

    checkpoint_pred_path = checkpoint_path.with_suffix(".txt")
    write_labels(str(checkpoint_pred_path), checkpoint_labels)
    print(f"{config['name']} Best Val Accuracy: {best_val_acc:.4f} (epoch {best_epoch})")
    print(f"{config['name']} Recomputed Train Accuracy: {train_acc:.4f}")
    print(f"{config['name']} Recomputed Val Accuracy: {val_acc:.4f}")
    print(f"Wrote checkpoint predictions to {checkpoint_pred_path}")

    return {
        "name": config["name"],
        "train_acc": train_acc,
        "val_acc": val_acc,
        "best_val_acc": best_val_acc,
        "best_epoch": best_epoch,
        "checkpoint": str(checkpoint_path),
        "prediction_path": str(checkpoint_pred_path),
        "val_prob": val_prob,
        "test_prob": test_prob,
        "config": {k: v for k, v in config.items() if k != "checkpoint"},
    }

## 6) Run Search and Select Final Result

This cell trains every config, compares validation-weighted soft voting with the best single model, and writes final `SEED.txt`.

In [16]:
results = [train_one_config(config, idx) for idx, config in enumerate(search_configs)]

weights = np.array([max(result["val_acc"] - ENSEMBLE_MIN_VAL_ACC, 0.0) ** ENSEMBLE_WEIGHT_POWER for result in results], dtype=np.float64)
if float(weights.sum()) <= 0.0:
    weights = np.ones(len(results), dtype=np.float64)
weights = weights / weights.sum()

weighted_val_prob = sum(weight * result["val_prob"] for weight, result in zip(weights, results))
weighted_test_prob = sum(weight * result["test_prob"] for weight, result in zip(weights, results))
ensemble_val_pred = np.argmax(weighted_val_prob, axis=1)
ensemble_test_pred = np.argmax(weighted_test_prob, axis=1)
ensemble_val_acc = accuracy(val_y, ensemble_val_pred)

best_single_idx = int(np.argmax([result["val_acc"] for result in results]))
best_single = results[best_single_idx]
best_single_val_acc = best_single["val_acc"]

if ensemble_val_acc >= best_single_val_acc:
    selected_method_name = "Validation-weighted Simple MoE vote"
    selected_val_acc = ensemble_val_acc
    selected_test_prob = weighted_test_prob
else:
    selected_method_name = best_single["name"]
    selected_val_acc = best_single_val_acc
    selected_test_prob = best_single["test_prob"]

selected_test_pred = np.argmax(selected_test_prob, axis=1)
all_test_labels = selected_test_pred.astype(int).tolist()
assert len(all_test_labels) == len(test_ds), f"Prediction count {len(all_test_labels)} != test sample count {len(test_ds)}"

write_labels(str(OUTPUT_PATH), all_test_labels)

serializable_results = []
for result in results:
    item = dict(result)
    item.pop("val_prob", None)
    item.pop("test_prob", None)
    serializable_results.append(item)

RESULTS_JSON_PATH.write_text(
    json.dumps(
        {
            "selected_method": selected_method_name,
            "selected_val_acc": selected_val_acc,
            "ensemble_val_acc": ensemble_val_acc,
            "best_single": {k: v for k, v in best_single.items() if k not in ("val_prob", "test_prob")},
            "weights": {result["name"]: float(weight) for result, weight in zip(results, weights)},
            "results": serializable_results,
        },
        indent=2,
    ),
    encoding="utf-8",
)

print("\nSimple MoE search summary")
for result_index, result in enumerate(results):
    marker = "*" if result is best_single else " "
    print(f"{marker} {result['name']}: train_acc={result['train_acc']:.4f}, val_acc={result['val_acc']:.4f}, best_epoch={result['best_epoch']}, ensemble_weight={weights[result_index]:.4f}")
print(f"Ensemble Val Accuracy: {ensemble_val_acc:.4f}")
print(f"Best Single Model: {best_single['name']} ({best_single_val_acc:.4f})")
print(f"Final selected method: {selected_method_name} | Val Accuracy: {selected_val_acc:.4f}")
print(f"Wrote final predictions to {OUTPUT_PATH}")
print(f"Wrote search results to {RESULTS_JSON_PATH}")


Training Simple MoE config: moe_h64_dp30_center
config: {'name': 'moe_h64_dp30_center', 'hidden_dim': 64, 'num_experts': 4, 'dropout': 0.3, 'lr': 0.0006, 'weight_decay': 0.001, 'label_smoothing': 0.05, 'scheduler_patience': 6}
Epoch [001/160] | Train Loss: 1.0972 | Train Acc: 0.3690 | Val Loss: 1.0901 | Val Acc: 0.4343 | Best: 0.4343 @ 001
Epoch [002/160] | Train Loss: 1.0802 | Train Acc: 0.4220 | Val Loss: 1.0789 | Val Acc: 0.4486 | Best: 0.4486 @ 002
Epoch [003/160] | Train Loss: 1.0655 | Train Acc: 0.4790 | Val Loss: 1.0707 | Val Acc: 0.4657 | Best: 0.4657 @ 003
Epoch [004/160] | Train Loss: 1.0541 | Train Acc: 0.5090 | Val Loss: 1.0673 | Val Acc: 0.4629 | Best: 0.4657 @ 003
Epoch [005/160] | Train Loss: 1.0418 | Train Acc: 0.5050 | Val Loss: 1.0591 | Val Acc: 0.4429 | Best: 0.4657 @ 003
Epoch [006/160] | Train Loss: 1.0328 | Train Acc: 0.5330 | Val Loss: 1.0543 | Val Acc: 0.4771 | Best: 0.4771 @ 006
Epoch [007/160] | Train Loss: 1.0229 | Train Acc: 0.5340 | Val Loss: 1.0508 | Val 